In [ ]:
# Resolve paths from the project root or any directory inside the project.
from pathlib import Path
import sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "02_Data" / "raw").is_dir() and (p / "03_Code").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter inside Chooser_Option_Research.")
sys.path.insert(0, str(PROJECT_ROOT / "03_Code"))

def save_raw_if_absent(frame, path):
    if path.exists():
        print(f"Preserved existing raw file: {path.name}")
    else:
        frame.to_csv(path, mode="x")

import yfinance as yf
import pandas as pd

In [56]:
#Download JPM historical stock price --test version
TICKER = "JPM"
START = "2024-01-01"
END = "2024-02-01"

jpm = yf.Ticker(TICKER)
df = jpm.history(start = START, end = END, auto_adjust = False, actions = True)
df.head()

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2024-01-02 00:00:00-05:00,169.089996,172.169998,168.910004,172.080002,161.472702,9977400,0.00,0.0
2024-01-03 00:00:00-05:00,171.860001,172.039993,170.369995,171.330002,160.768936,9852300,0.00,0.0
2024-01-04 00:00:00-05:00,170.639999,173.350006,170.539993,171.410004,161.835815,11972500,1.05,0.0
2024-01-05 00:00:00-05:00,171.470001,173.380005,171.470001,172.270004,162.647812,10066000,0.00,0.0
2024-01-08 00:00:00-05:00,172.020004,172.360001,169.490005,172.020004,162.411743,11229900,0.00,0.0


In [57]:
print(len(df))
print(df.columns)
print(df.index.min())
print(df.index.max())

21
Index(['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends',
       'Stock Splits'],
      dtype='str')
2024-01-02 00:00:00-05:00
2024-01-31 00:00:00-05:00


In [58]:
#Actual codes start here
START = "2018-01-01"
END = "2025-01-01"

In [59]:
#Write a reusable API function to extract data
def stock_data(ticker, start_date, end_date):
    stock = yf.Ticker(ticker)
    data = stock.history(start = start_date, end = end_date, auto_adjust = False, actions = True)
    return data

In [6]:
#check data validity
jpm_data = stock_data(TICKER, START, END)
jpm_data.head()
print(len(jpm_data))
print(jpm_data.columns.tolist())
print(jpm_data.index.min())
print(jpm_data.index.max())

1761
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']
2018-01-02 00:00:00-05:00
2024-12-31 00:00:00-05:00


In [7]:
TICKER = "^VIX"
vix = stock_data(TICKER, START, END)
vix.head()
print(len(vix))
print(vix.columns.tolist())
print(vix.index.min())
print(vix.index.max())

1761
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']
2018-01-02 00:00:00-06:00
2024-12-31 00:00:00-06:00


In [ ]:
save_raw_if_absent(jpm_data, PROJECT_ROOT / "02_Data/raw/JPM_daily_2018_2024.csv")
save_raw_if_absent(vix, PROJECT_ROOT / "02_Data/raw/VIX_daily_2018_2024.csv")

In [ ]:
check1 = pd.read_csv(PROJECT_ROOT / "02_Data/raw/JPM_daily_2018_2024.csv")
check2 = pd.read_csv(PROJECT_ROOT / "02_Data/raw/VIX_daily_2018_2024.csv")
print(check1.shape)
print(check2.shape)
print(check1.isna().sum())
print(check2.isna().sum())

In [ ]:
#API Set up--completed in terminal
import os
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
fred_api_key = os.getenv("FRED_API_KEY")

In [11]:
#get HTTPS data (DGS1 & DGS6MO from FRED)
import requests
import pandas as pd
#retrieved observations for a specified economic data series
url = "https://api.stlouisfed.org/fred/series/observations" 

In [12]:
#set up API Parameters & get DGS1 data
#For FRED, end date is inclusive, should be 2024-12-31. yf is the opposite
para = {"series_id":"DGS1", "api_key":fred_api_key, "file_type":"json", "observation_start":"2018-01-01", "observation_end":"2024-12-31"} 

In [13]:
#Send HTTP GET Request
res = requests.get(url, params = para, timeout = 30)
res.raise_for_status() #check if the data return successfully
print(res.status_code)

200


In [14]:
#get the JSON response (the info we need is in "observations"
data = res.json()
#transfer JSON into df
dgs1_df = pd.DataFrame(data["observations"])
dgs1_df.head()

,realtime_start,realtime_end,date,value
0,2026-10-11,2026-10-11,2018-01-01,.
1,2026-10-11,2026-10-11,2018-01-02,1.83
2,2026-10-11,2026-10-11,2018-01-03,1.81
3,2026-10-11,2026-10-11,2018-01-04,1.82
4,2026-10-11,2026-10-11,2018-01-05,1.80


In [15]:
#check data (col & row, start & end, empty data)
print(dgs1_df.shape)
print(dgs1_df["date"].min())
print(dgs1_df["date"].max())
print((dgs1_df["value"] == ".").sum())

(1827, 4)
2018-01-01
2024-12-31
77


In [ ]:
save_raw_if_absent(dgs1_df, PROJECT_ROOT / "02_Data/raw/DGS1_2018_2024.csv")

In [17]:
#get DGS6MO data, identical codes
para["series_id"] = "DGS6MO"
res = requests.get(url, params = para, timeout = 30)
data = res.json()
dgs6mo_df = pd.DataFrame(data["observations"])
dgs6mo_df.head()

,realtime_start,realtime_end,date,value
0,2026-10-11,2026-10-11,2018-01-01,.
1,2026-10-11,2026-10-11,2018-01-02,1.61
2,2026-10-11,2026-10-11,2018-01-03,1.59
3,2026-10-11,2026-10-11,2018-01-04,1.60
4,2026-10-11,2026-10-11,2018-01-05,1.58


In [18]:
print(dgs6mo_df.shape)
print(dgs6mo_df["date"].min())
print(dgs6mo_df["date"].max())
print((dgs6mo_df["value"] == ".").sum())

(1827, 4)
2018-01-01
2024-12-31
77


In [ ]:
save_raw_if_absent(dgs6mo_df, PROJECT_ROOT / "02_Data/raw/DGS6MO_2018_2024.csv")

In [ ]:
#set up Alphavantage API Key for stock data backup and Cross-validation
import os
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
av_api_key = os.getenv("ALPHAVANTAGE_API_KEY")

In [25]:
#Similarly
av_url = "https://www.alphavantage.co/query"
av_para = {"function":"TIME_SERIES_DAILY", "symbol":"JPM", "outputsize":"compact", "apikey":av_api_key}
av_res = requests.get(av_url, params=av_para, timeout=30)
av_res.raise_for_status()
print(av_res.status_code)
av_data = av_res.json()

200


In [26]:
print(av_data.keys())

dict_keys(['Meta Data', 'Time Series (Daily)'])


In [27]:
#Unlike FRED's data is a list of dict, AV is a dist of dist
#so we must tranpose it, to make the date becomes the index
av_df = pd.DataFrame(av_data["Time Series (Daily)"]).T
av_df.head()

,1. open,2. high,3. low,4. close,5. volume
2026-10-09,331.2000,333.5400,330.5800,332.9900,5058747
2026-10-08,327.2500,332.3100,325.6250,331.4200,8069273
2026-10-07,327.2000,329.8699,325.0700,329.5800,7005558
2026-10-06,332.7300,333.7900,330.9300,331.2800,6289788
2026-10-05,333.3100,333.7500,330.5900,332.3800,5346072


In [28]:
print(av_df.shape)
print(av_df.index.min())
print(av_df.index.max())

(100, 5)
2026-05-19
2026-10-09


In [ ]:
save_raw_if_absent(av_df, PROJECT_ROOT / "02_Data/raw/JPM_AlphaVantage_daily_latest_100_days.csv")

In [5]:
#Cross-source data validation, we use Nasdaq data in comparision to yf

,Open,High,Low,Close,Volume
Date,,,,,
2023-06-05 00:00:00-04:00,140.110001,140.110001,138.130005,139.089996,8511900


In [ ]:
# Yahoo Finance Data
yf_df = pd.read_csv(
    PROJECT_ROOT / "02_Data/raw/JPM_daily_2018_2024.csv"
)

# Nasdaq Data
nasdaq_df = pd.read_csv(
    PROJECT_ROOT / "02_Data/raw/JPM_Nasdaq_2016_2026.csv"
)

In [10]:
print(yf_df.shape)
print(nasdaq_df.shape)

(1761, 9)
(2513, 6)


In [11]:
print(yf_df.columns.tolist())
print(nasdaq_df.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']
['Date', 'Close/Last', 'Volume', 'Open', 'High', 'Low']


In [15]:
#two csv have different date format
#yf: 2023-06-05 00:00:00-04:00
#nasdaq: 06/05/2023
print(yf_df["Date"])
print(nasdaq_df["Date"])

0       2018-01-02 00:00:00-05:00
1       2018-01-03 00:00:00-05:00
2       2018-01-04 00:00:00-05:00
3       2018-01-05 00:00:00-05:00
4       2018-01-08 00:00:00-05:00
                  ...            
1756    2024-12-24 00:00:00-05:00
1757    2024-12-26 00:00:00-05:00
1758    2024-12-27 00:00:00-05:00
1759    2024-12-30 00:00:00-05:00
1760    2024-12-31 00:00:00-05:00
Name: Date, Length: 1761, dtype: str
0       10/09/2026
1       10/08/2026
2       10/07/2026
3       10/06/2026
4       10/05/2026
           ...    
2508    10/17/2016
2509    10/14/2016
2510    10/13/2016
2511    10/12/2016
2512    10/11/2016
Name: Date, Length: 2513, dtype: str


In [25]:
#Data Processing
yf_df["Date"] = yf_df["Date"].str[:10]
nasdaq_df["Date"] = pd.to_datetime(nasdaq_df["Date"])
yf_df["Date"] = pd.to_datetime(yf_df["Date"].str[:10])

In [26]:
print(yf_df["Date"])
print(nasdaq_df["Date"])

0      2018-01-02
1      2018-01-03
2      2018-01-04
3      2018-01-05
4      2018-01-08
          ...    
1756   2024-12-24
1757   2024-12-26
1758   2024-12-27
1759   2024-12-30
1760   2024-12-31
Name: Date, Length: 1761, dtype: datetime64[us]
0      2026-10-09
1      2026-10-08
2      2026-10-07
3      2026-10-06
4      2026-10-05
          ...    
2508   2016-10-17
2509   2016-10-14
2510   2016-10-13
2511   2016-10-12
2512   2016-10-11
Name: Date, Length: 2513, dtype: datetime64[us]


In [ ]:
#Extract 18-24 data from Nasdaq
nasdaq_2018_2024 = nasdaq_df[nasdaq_df["Date"].dt.year.between(2018, 2024)]
print(nasdaq_2018_2024.shape)
#1761 is consistent with the number shown in yf
# Normalize provider price/volume strings in memory.
nasdaq_2018_2024 = nasdaq_2018_2024.copy()
for column in ["Open", "High", "Low", "Close/Last", "Volume"]:
    nasdaq_2018_2024[column] = pd.to_numeric(nasdaq_2018_2024[column].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False))


In [28]:
#Cross-validation
merged = pd.merge(
    yf_df,
    nasdaq_2018_2024,
    on="Date",
    suffixes=("_yf", "_nasdaq")
)

print(merged.shape)
merged.head()

(1761, 14)


,Date,Open_yf,High_yf,Low_yf,Close,Adj Close,Volume_yf,Dividends,Stock Splits,Close/Last,Volume_nasdaq,Open_nasdaq,High_nasdaq,Low_nasdaq
0,2018-01-02,107.629997,108.019997,106.809998,107.949997,85.091499,13578800,0.00,0.0,$107.95,13571680,$107.63,$108.02,$106.8056
1,2018-01-03,107.860001,108.489998,107.480003,108.059998,85.178230,11901000,0.00,0.0,$108.06,11895160,$107.86,$108.49,$107.48
2,2018-01-04,108.360001,110.029999,108.199997,109.040001,86.398468,12953700,0.56,0.0,$109.04,12951630,$108.36,$110.03,$108.20
3,2018-01-05,109.260002,109.550003,107.779999,108.339996,85.843803,14155000,0.00,0.0,$108.34,14144400,$109.26,$109.55,$107.775
4,2018-01-08,108.150002,108.680000,107.699997,108.500000,85.970566,12466500,0.00,0.0,$108.50,12466000,$108.15,$108.68,$107.70


In [37]:
#calculate the absolute difference between HIGH & collect the abnormalies
merged["High_diff"] = abs(merged["High_yf"] - merged["High_nasdaq"])
high_errors = merged[merged["High_diff"] > 0.01]
print("Number of discrepancies:", len(high_errors))
high_errors[["Date", "High_yf", "High_nasdaq", "High_diff"]].head()

Number of discrepancies: 1


,Date,High_yf,High_nasdaq,High_diff
1364,2023-06-05,140.110001,139.31,0.800001


In [ ]:
merged["Volume_diff"] = abs(merged["Volume_yf"] - merged["Volume_nasdaq"])
#Similarly, for open, low, closed, and volume
merged["Low_diff"] = abs(merged["Low_yf"] - merged["Low_nasdaq"])
low_errors = merged[merged["Low_diff"] > 0.01]
print("Number of discrepancies:", len(low_errors))

merged["Open_diff"] = abs(merged["Open_yf"] - merged["Open_nasdaq"])
open_errors = merged[merged["Open_diff"] > 0.01]
print("Number of discrepancies:", len(open_errors))
display(open_errors[["Date", "Open_yf", "Open_nasdaq", "Open_diff"]].head())

merged["Close_diff"] = abs(merged["Close"] - merged["Close/Last"])
close_errors = merged[merged["Close_diff"] > 0.01]
print("Number of discrepancies:", len(close_errors))

volume_errors = merged[merged["Volume_diff"] > 0]
print("Number of discrepancies:", len(volume_errors))
display(volume_errors[["Date", "Volume_yf", "Volume_nasdaq", "Volume_diff"]].head())
#find the largest 5 diff
volume_errors.nlargest(5, "Volume_diff")[
    ["Date", "Volume_yf", "Volume_nasdaq", "Volume_diff"]
]

In [54]:
merged.loc[
    merged["Date"].isin(pd.to_datetime(["2023-01-24", "2023-06-05"])),
    ["Date", "Open_yf", "Open_nasdaq",
     "High_yf", "High_nasdaq",
     "Low_yf", "Low_nasdaq",
     "Close", "Close/Last"]
]


,Date,Open_yf,Open_nasdaq,High_yf,High_nasdaq,Low_yf,Low_nasdaq,Close,Close/Last
1273,2023-01-24,136.419998,140.29,140.289993,140.29,135.000000,135.00,138.449997,138.45
1364,2023-06-05,140.110001,140.11,140.110001,139.31,138.130005,138.13,139.089996,139.09


In [61]:
#Reusable API function--for yf we use the stock_data fn above
#for testing
test_data = stock_data("JPM", "2024-01-01", "2024-02-01")
print(test_data.shape)
test_data.head()

(21, 8)


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2024-01-02 00:00:00-05:00,169.089996,172.169998,168.910004,172.080002,161.472702,9977400,0.00,0.0
2024-01-03 00:00:00-05:00,171.860001,172.039993,170.369995,171.330002,160.768936,9852300,0.00,0.0
2024-01-04 00:00:00-05:00,170.639999,173.350006,170.539993,171.410004,161.835815,11972500,1.05,0.0
2024-01-05 00:00:00-05:00,171.470001,173.380005,171.470001,172.270004,162.647812,10066000,0.00,0.0
2024-01-08 00:00:00-05:00,172.020004,172.360001,169.490005,172.020004,162.411743,11229900,0.00,0.0


In [ ]:
#for FRED, recall the task we did on day 3
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

fred_api_key = os.getenv("FRED_API_KEY")

def fred_data(series_id, start_date, end_date):
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": fred_api_key,
        "file_type": "json",
        "observation_start": start_date,
        "observation_end": end_date
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = pd.DataFrame(response.json()["observations"])

    return data

In [67]:
#for testing
dgs1_test = fred_data("DGS1", "2018-01-01", "2024-12-31")
print(dgs1_test.shape)
print(dgs1_test.head())

dgs6mo_test = fred_data("DGS6MO", "2018-01-01", "2024-12-31")
print(dgs6mo_test.shape)
dgs6mo_test.head()

(1827, 4)
  realtime_start realtime_end        date value
0     2026-10-11   2026-10-11  2018-01-01     .
1     2026-10-11   2026-10-11  2018-01-02  1.83
2     2026-10-11   2026-10-11  2018-01-03  1.81
3     2026-10-11   2026-10-11  2018-01-04  1.82
4     2026-10-11   2026-10-11  2018-01-05  1.80
(1827, 4)


,realtime_start,realtime_end,date,value
0,2026-10-11,2026-10-11,2018-01-01,.
1,2026-10-11,2026-10-11,2018-01-02,1.61
2,2026-10-11,2026-10-11,2018-01-03,1.59
3,2026-10-11,2026-10-11,2018-01-04,1.60
4,2026-10-11,2026-10-11,2018-01-05,1.58


In [ ]:
from src.W01_data_extraction import stock_data, fred_data
print("Functions imported successfully:", callable(stock_data), callable(fred_data))
